# Switchyard Business Client

This notebook verifies the Switchyard endpoint and builds a simple LangChain customer-request triage app. The app sends every request to the public Switchyard route named `switchyard`; Switchyard selects the efficient or capable Foundry deployment configured for this participant.

## 1. Install the LangChain OpenAI integration

Switchyard exposes an OpenAI-compatible Chat Completions API, so the app uses LangChain's `ChatOpenAI` integration with a custom base URL.

In [ ]:
%pip install -qU langchain-openai

## 2. Connect to the participant's Container App

The setup first reads `SWITCHYARD_FQDN` from the notebook environment. If it is not set, it resolves the hostname with Azure CLI using `RESOURCE_GROUP` and `SWITCHYARD_APP_NAME`. It similarly reuses `AZURE_AI_AUTH_TOKEN` when present or obtains a short-lived token for the Azure identity currently signed in to the CLI. That identity must have data-plane access to the participant's Foundry resource.

In [ ]:
import os
import subprocess


def run_az(*arguments: str) -> str:
    """Run Azure CLI and return a non-empty text result."""
    result = subprocess.run(
        ["az", *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if not result:
        raise RuntimeError(f"Azure CLI returned no value for: az {' '.join(arguments)}")
    return result


switchyard_fqdn = os.getenv("SWITCHYARD_FQDN", "").strip()
if not switchyard_fqdn:
    resource_group = os.getenv("RESOURCE_GROUP", "").strip()
    switchyard_app_name = os.getenv("SWITCHYARD_APP_NAME", "").strip()
    if not resource_group or not switchyard_app_name:
        raise RuntimeError(
            "Set SWITCHYARD_FQDN, or set both RESOURCE_GROUP and "
            "SWITCHYARD_APP_NAME, before running this cell."
        )
    switchyard_fqdn = run_az(
        "containerapp",
        "show",
        "--name",
        switchyard_app_name,
        "--resource-group",
        resource_group,
        "--query",
        "properties.configuration.ingress.fqdn",
        "--output",
        "tsv",
    )

azure_ai_auth_token = os.getenv("AZURE_AI_AUTH_TOKEN", "").strip()
if not azure_ai_auth_token:
    azure_ai_auth_token = run_az(
        "account",
        "get-access-token",
        "--resource",
        "https://cognitiveservices.azure.com",
        "--query",
        "accessToken",
        "--output",
        "tsv",
    )

# Child processes, including the Bash smoke test below, inherit these values.
os.environ["SWITCHYARD_FQDN"] = switchyard_fqdn
os.environ["AZURE_AI_AUTH_TOKEN"] = azure_ai_auth_token

switchyard_base_url = f"https://{switchyard_fqdn}/v1"
print(f"Switchyard endpoint: {switchyard_base_url}")
print("Azure access token acquired; its value is intentionally hidden.")

## 3. Verify the endpoint with curl

This is the same workstation test used in the lab README. A successful response confirms that external ingress, forwarded authentication, the Switchyard route, and the upstream Foundry deployment are working.

In [ ]:
%%bash
: "${SWITCHYARD_FQDN:?Set SWITCHYARD_FQDN before running this cell.}"
: "${AZURE_AI_AUTH_TOKEN:?Set AZURE_AI_AUTH_TOKEN before running this cell.}"

curl --fail-with-body --silent --show-error \
  --request POST \
  "https://${SWITCHYARD_FQDN}/v1/chat/completions" \
  --header "Authorization: Bearer $AZURE_AI_AUTH_TOKEN" \
  --header "Content-Type: application/json" \
  --data '{
    "model": "switchyard",
    "messages": [
      {
        "role": "user",
        "content": "Explain model routing in two sentences."
      }
    ]
  }'


## 4. Build a LangChain customer-request triage app

The chain turns an incoming customer message into an action-oriented business brief. It identifies urgency and ownership, summarizes the issue, proposes next steps, and drafts a response. The output remains plain Markdown so it works with model deployments that do not support tool calling or native structured output.

In [ ]:
from IPython.display import Markdown, display
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    model="switchyard",
    base_url=switchyard_base_url,
    api_key=azure_ai_auth_token,
    use_responses_api=False,
    temperature=0,
)

triage_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a customer-operations analyst. Triage incoming business requests
for a human support team. Do not invent policies, commitments, refunds, or
technical facts. Clearly label any assumptions.

Return concise Markdown with exactly these headings:
## Priority
Choose Low, Medium, High, or Critical and give one-sentence reasoning.
## Owning team
Choose Sales, Billing, Technical Support, Security, or Customer Success.
## Summary
Summarize the request in no more than three bullets.
## Recommended actions
Give a numbered list of the next actions for the business.
## Draft response
Write a concise, professional response to the customer.""",
        ),
        (
            "human",
            """Customer: {customer_name}
Account tier: {account_tier}
Request received through: {channel}

Customer message:
{customer_message}""",
        ),
    ]
)

triage_chain = triage_prompt | model | StrOutputParser()


def triage_customer_request(
    customer_name: str,
    account_tier: str,
    channel: str,
    customer_message: str,
) -> str:
    """Analyze one customer request through the Switchyard route."""
    return triage_chain.invoke(
        {
            "customer_name": customer_name,
            "account_tier": account_tier,
            "channel": channel,
            "customer_message": customer_message,
        }
    )

## 5. Run the business app

Edit the sample values to try routine and complex requests. Switchyard can route each call to the model deployment best suited to the request.

In [ ]:
result = triage_customer_request(
    customer_name="Contoso Retail",
    account_tier="Enterprise",
    channel="Email",
    customer_message=(
        "Our production order-processing integration began returning intermittent "
        "timeouts after yesterday's deployment. Approximately 20% of orders are "
        "affected. We rolled back our application, but the errors continue. Please "
        "help us restore service and advise what diagnostic information you need."
    ),
)

display(Markdown(result))